# 08 — Problem 2: Low-Review Risk at Delivery

**Question.** Will this delivered order draw a 1–2 star review?
**Stakeholder.** Customer experience — flag an at-risk order for outreach *before*
the review is written.
**Prediction moment.** `at_delivery`. The parcel has arrived, so how delivery went
is history and admissible; the review and its timestamps are still forbidden.

Phase 1 measured what that moment is worth: the same target predicted at checkout
reaches PR-AUC 0.212, and at delivery 0.402. The prediction moment nearly doubles
the achievable signal.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, evaluate as E, modeling as M
from data_load import load_all, load_analysis
from features import *
from viz import use_report_style, save_fig

use_report_style()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "report" / "results"
def results(name):
    """Read one canonical result table written by src/train_models.py."""
    p = RESULTS / name
    assert p.exists(), f"{p} missing - run `python src/train_models.py` first"
    return pd.read_csv(p)
print("seed", config.SEED, "| window", config.ANALYSIS_START.date(), "->", config.ANALYSIS_END.date())

seed 42 | window 2017-01-01 -> 2018-08-31


In [2]:
low = results("clf_p2.csv")
display(low[["model", "family", "rung", "cv_pr_auc", "val_pr_auc", "roc_auc",
             "pr_auc", "pr_auc_lift", "balanced_accuracy", "precision", "recall",
             "f1", "accuracy", "majority_accuracy", "brier", "threshold"]].round(3))

,model,family,rung,cv_pr_auc,val_pr_auc,roc_auc,pr_auc,pr_auc_lift,balanced_accuracy,precision,recall,f1,accuracy,majority_accuracy,brier,threshold
0,LogisticRegression,Linear,0,0.465,0.320,0.659,0.291,2.634,0.636,0.328,0.365,0.345,0.847,0.89,0.090,0.17
1,LogisticRegression (tuned L2),Linear,1,0.466,0.320,0.659,0.291,2.634,0.636,0.317,0.371,0.342,0.843,0.89,0.090,0.16
2,DecisionTreeClassifier,Tree-based,0,0.543,0.389,0.722,0.396,3.588,0.677,0.562,0.391,0.461,0.899,0.89,0.080,0.25
3,RandomForestClassifier,Tree-based,1,0.568,0.403,0.744,0.405,3.667,0.680,0.576,0.396,0.470,0.901,0.89,0.078,0.27
4,HistGradientBoostingClassifier,Tree-based,2,0.568,0.407,0.749,0.415,3.761,0.680,0.579,0.396,0.471,0.902,0.89,0.078,0.25
5,"VotingClassifier (A+B, soft)",Ensemble,0,0.561,0.404,0.747,0.407,3.693,0.683,0.477,0.422,0.448,0.885,0.89,0.079,0.20
6,StackingClassifier (A+B),Ensemble,1,0.568,0.409,0.749,0.414,3.751,0.681,0.575,0.398,0.470,0.901,0.89,0.078,0.19


## 8.1 The ladder

In [3]:
for fam in ("Linear", "Tree-based", "Ensemble"):
    part = low[low["family"] == fam].sort_values("rung")
    if part.empty:
        continue
    print(f"\n{fam}")
    for _, r in part.iterrows():
        print(f"  rung {int(r['rung'])}  {r['model']:36s} cvPR {r['cv_pr_auc']:.3f}"
              f"   valPR {r['val_pr_auc']:.3f}   testPR {r['pr_auc']:.3f}"
              f"   ROC {r['roc_auc']:.3f}   balacc {r['balanced_accuracy']:.3f}")

lin = low[low['family'] == 'Linear']['pr_auc'].max()
tree = low[low['family'] == 'Tree-based']['pr_auc'].max()
print(f"\ntree-based over linear: x{tree/lin:.2f} on PR-AUC")


Linear
  rung 0  LogisticRegression                   cvPR 0.465   valPR 0.320   testPR 0.291   ROC 0.659   balacc 0.636
  rung 1  LogisticRegression (tuned L2)        cvPR 0.466   valPR 0.320   testPR 0.291   ROC 0.659   balacc 0.636

Tree-based
  rung 0  DecisionTreeClassifier               cvPR 0.543   valPR 0.389   testPR 0.396   ROC 0.722   balacc 0.677
  rung 1  RandomForestClassifier               cvPR 0.568   valPR 0.403   testPR 0.405   ROC 0.744   balacc 0.680
  rung 2  HistGradientBoostingClassifier       cvPR 0.568   valPR 0.407   testPR 0.415   ROC 0.749   balacc 0.680

Ensemble
  rung 0  VotingClassifier (A+B, soft)         cvPR 0.561   valPR 0.404   testPR 0.407   ROC 0.747   balacc 0.683
  rung 1  StackingClassifier (A+B)             cvPR 0.568   valPR 0.409   testPR 0.414   ROC 0.749   balacc 0.681

tree-based over linear: x1.43 on PR-AUC


Note where most of the gain arrives. A single depth-limited decision tree
already captures the bulk of it — the signal in this problem is close to a
threshold rule on how late the parcel was, which is exactly what section 8.5
shows. Boosting adds a real but modest increment on top.

## 8.2 Imbalance: the ablation Phase 1 promised

Phase 1's literature review argued, from van den Goorbergh et al. (2022), that
correcting class imbalance does not improve discrimination and badly miscalibrates
the resulting probabilities — and committed Phase 2 to testing it rather than
assuming it. This is that test, on the selected model, on the test window.

In [4]:
abl = results("weight_ablation.csv")
display(abl[["model", "roc_auc", "pr_auc", "brier", "mean_predicted",
             "actual_rate"]].round(4))

w = abl[abl["model"].str.contains("balanced")].iloc[0]
u = abl[~abl["model"].str.contains("balanced")].iloc[0]
print(f"\nweighting the loss:")
print(f"  discrimination : ROC {u['roc_auc']:.3f} -> {w['roc_auc']:.3f}  "
      f"(no meaningful gain)")
print(f"  calibration    : Brier {u['brier']:.4f} -> {w['brier']:.4f}  "
      f"({100*(w['brier']/u['brier']-1):.0f}% worse)")
print(f"  mean predicted risk {w['mean_predicted']:.3f} against an actual "
      f"{w['actual_rate']:.3f} - risk overstated {w['mean_predicted']/w['actual_rate']:.1f}x")
print("\nSo: fit on the observed prior, then move the decision threshold.")

,model,roc_auc,pr_auc,brier,mean_predicted,actual_rate
0,fit on the observed prior,0.7491,0.4148,0.0778,0.1193,0.1103
1,class_weight='balanced',0.7473,0.4114,0.1331,0.3374,0.1103



weighting the loss:
  discrimination : ROC 0.749 -> 0.747  (no meaningful gain)
  calibration    : Brier 0.0778 -> 0.1331  (71% worse)
  mean predicted risk 0.337 against an actual 0.110 - risk overstated 3.1x

So: fit on the observed prior, then move the decision threshold.


## 8.3 Calibration

The scores are used two ways — to rank orders for outreach, and to multiply by a
cost. Ranking needs only monotonicity; the cost calculation needs the numbers to
mean what they say, so calibration is reported rather than assumed.

In [5]:
cal = results("calibration_p2.csv")
display(cal.round(4))
print(f"largest gap between predicted risk and observed rate: "
      f"{cal['gap'].abs().max():.3f}")

import figures_phase2 as F
print(Path(F.fig_calibration()).name)

,bin,n,mean_predicted,observed_rate,gap
0,0.0-0.1,10525,0.0769,0.0639,0.0129
1,0.1-0.2,1367,0.1269,0.1361,-0.0091
2,0.2-0.3,111,0.2507,0.2973,-0.0466
3,0.3-0.4,143,0.3520,0.3986,-0.0466
4,0.4-0.5,95,0.4486,0.5053,-0.0566
5,0.5-0.6,86,0.5537,0.5698,-0.0161
6,0.6-0.7,68,0.6435,0.6471,-0.0036
7,0.7-0.8,297,0.7581,0.7104,0.0476
8,0.8-0.9,163,0.8282,0.7178,0.1104


largest gap between predicted risk and observed rate: 0.110


fig14_calibration.pdf


## 8.4 The operating point

The threshold is chosen on the **validation** period by maximising F1, then
applied unchanged to the test window. Choosing it on the test window would be a
subtler form of the leakage this project spends its effort avoiding.

In [6]:
import json
meta = json.loads((config.MODELS_DIR / "model_metadata.json").read_text("utf-8"))
thr = meta["p2"]["threshold"]
sweep = results("threshold_sweep_p2.csv")
at = lambda t: sweep.iloc[(sweep["threshold"] - t).abs().idxmin()]
d, t_ = at(0.5), at(thr)
display(pd.DataFrame([d, t_], index=[f"default 0.50", f"tuned {thr:.2f}"])
        [["precision", "recall", "f1", "balanced_accuracy", "flagged_pct"]].round(3))
print(Path(F.fig_threshold_and_cost()).name)
print(Path(F.fig_low_review_curves()).name)

,precision,recall,f1,balanced_accuracy,flagged_pct
default 0.50,0.686,0.297,0.414,0.640,4.776
tuned 0.19,0.575,0.398,0.470,0.681,7.631


fig15_threshold_cost.pdf


fig13_low_review.pdf


## 8.5 Does it generalise to buyers it has never seen?

Phase 1 warned that a temporal cut can still put one of a repeat buyer's orders in
training and another in test. The clean test subset removes those customers
entirely; if the model were memorising buyers rather than learning the problem,
performance would fall here.

In [7]:
p2 = meta["p2"]
print(f"all test orders            : PR-AUC {p2['test_pr_auc']:.3f}")
print(f"customers unseen in training: PR-AUC {p2['unseen_customers_pr_auc']:.3f} "
      f"({p2['unseen_customers_n']:,} orders)")
gap = abs(p2["test_pr_auc"] - p2["unseen_customers_pr_auc"])
print(f"gap {gap:.3f} - the model is not memorising buyers")
assert gap < 0.05, "performance depends on having seen the customer before"

all test orders            : PR-AUC 0.414
customers unseen in training: PR-AUC 0.415 (12,617 orders)
gap 0.002 - the model is not memorising buyers
